# Content Bot · Giọng đọc AI trên Kaggle

1. Tạo notebook **private**, chọn GPU nếu dùng `cuda`. Kiểm tra quota trong giao diện Kaggle.
2. Upload `voiceover-kaggle.zip` thành input riêng tư, sửa INPUT bên dưới. Gói có lời đọc/mẫu giọng; không chứa phim.
3. Chạy các cell cài đặt môi trường và cell nghe thử một đoạn đầu tiên để kiểm tra chất lượng giọng trước khi chạy toàn bộ lô.
4. Lưu **Save Version với outputs** hoặc tải `voiceover-results.zip` về trước khi kết thúc phiên. File trong phiên chưa lưu có thể mất.
5. Để tiếp tục, thêm gói kết quả cũ vào input và đặt PREVIOUS_RESULTS. Chỉ xử lý đoạn thiếu hoặc checksum sai.
6. Nhập ZIP kết quả vào Content Bot → Giọng đọc → Kaggle và tải audio. Không chỉnh manifest.


In [ ]:
from pathlib import Path
import json, zipfile, shutil, sys, subprocess, os
INPUT = Path("/kaggle/input/voiceover/voiceover-kaggle.zip")
PREVIOUS_RESULTS = None  # Path("/kaggle/input/previous/voiceover-results.zip")
if "process" in globals() and process.poll() is None:
    raise RuntimeError("Worker vẫn đang chạy. Đợi hoặc tạm dừng trước khi mở gói khác.")
with zipfile.ZipFile(INPUT) as package:
    package_manifest = json.loads(package.read("manifest.json"))
import hashlib
package_key = hashlib.sha256(json.dumps(package_manifest, sort_keys=True).encode()).hexdigest()[:20]
ROOT = (Path("/kaggle/working/content-bot-voice") / package_key).resolve()
ROOT.mkdir(parents=True, exist_ok=True)
def unpack(path, allowed):
    with zipfile.ZipFile(path) as z:
        if len(z.infolist()) > 40020 or len(set(z.namelist())) != len(z.namelist()):
            raise ValueError("Số file hoặc tên file trùng không hợp lệ")
        if sum(i.file_size for i in z.infolist()) > 4 * 1024**3:
            raise ValueError("Gói quá lớn")
        for i in z.infolist():
            target = (ROOT / i.filename).resolve()
            if not target.is_relative_to(ROOT) or i.is_dir():
                raise ValueError("Đường dẫn gói không hợp lệ")
            if i.filename not in allowed and not __import__("re").fullmatch(r"assets/[a-f0-9]{64}\.(wav|json)", i.filename):
                raise ValueError("Loại file không hợp lệ")
        z.extractall(ROOT)
unpack(INPUT, {"manifest.json", "worker.py", "requirements.txt", "requirements-gpu.txt", "reference.wav", "voiceover_kaggle.ipynb"})
manifest = json.loads((ROOT / "manifest.json").read_text())
if PREVIOUS_RESULTS:
    with zipfile.ZipFile(PREVIOUS_RESULTS) as z:
        if json.loads(z.read("manifest.json")) != manifest:
            raise ValueError("Checkpoint không thuộc cùng gói tác vụ")
    unpack(PREVIOUS_RESULTS, {"manifest.json", "progress.json", "environment.txt"})
(ROOT / "control.json").unlink(missing_ok=True)
print("Số đoạn:", len(manifest["clips"]), "Thiết bị:", manifest["device"])


In [ ]:
if "process" in globals() and process.poll() is None:
    raise RuntimeError("Đợi worker kết thúc trước khi thay môi trường.")
VENV = ROOT / "runtime"
PYTHON = str(VENV / "bin" / "python")
if not Path(PYTHON).is_file():
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")], check=True)
if manifest["device"] == "cuda":
    subprocess.run([PYTHON, "-m", "pip", "install", "torch==2.8.0", "torchaudio==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(ROOT / "requirements-gpu.txt")], check=True)
    subprocess.run([PYTHON, "-c", "import torch; assert torch.cuda.is_available(), 'Bật GPU trong Settings'; x=torch.ones((64,64),device='cuda'); assert torch.isfinite(x@x).all().item(); print(torch.cuda.get_device_name(0))"], check=True)

subprocess.run([PYTHON, "-m", "pip", "check"], check=True)
with (ROOT / "environment.txt").open("w") as environment:
    subprocess.run([PYTHON, "-m", "pip", "freeze"], stdout=environment, check=True)


### Nghe thử một đoạn trước khi chạy toàn bộ

Chạy cell dưới để nạp model, tạo thử đoạn đầu tiên và phát audio trực tiếp trong notebook. Bạn có thể nghe thử âm sắc, ngắt câu và tốc độ đọc trước khi bấm chạy toàn bộ lô.


In [ ]:
import os, subprocess
from IPython.display import Audio, display

if "process" in globals() and process.poll() is None:
    raise RuntimeError("Worker đang chạy. Đợi hoặc tạm dừng trước khi nghe thử.")

first_clip = manifest["clips"][0]
print("=== NGHE THỬ ĐOẠN ĐẦU TIÊN ===")
print("Clip ID:", first_clip["id"])
print("Lời đọc:", first_clip["text"])

preview_code = '''
import sys, json, os
from pathlib import Path
root = Path(sys.argv[1])
sys.path.insert(0, str(root))
manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
clip = manifest["clips"][0]
import worker
engine = worker.load_engine(manifest["device"], model_id=manifest["profile"].get("model_id", worker.MODEL), download=os.environ.get("VOICE_ALLOW_DOWNLOAD") == "1")
voice_args = worker.voice_arguments(engine, manifest["profile"], root)
audio = worker.infer_with_retry(engine, clip["text"], voice_args, manifest["device"])
import soundfile as sf
out_wav = root / "preview_sample.wav"
sf.write(str(out_wav), audio, engine.sample_rate, subtype="PCM_16", format="WAV")
print("Đã tạo mẫu nghe thử thành công:", out_wav)
'''

MODEL_CACHE = Path("/kaggle/working/voice-models")
env = os.environ.copy()
env["CONTENT_BOT_VOICE_MODEL_DIR"] = str(MODEL_CACHE)
env["VOICE_ALLOW_DOWNLOAD"] = "1"
env["PYTHONUTF8"] = "1"

subprocess.run([PYTHON, "-c", preview_code, str(ROOT)], env=env, check=True)
preview_audio_file = ROOT / "preview_sample.wav"
if preview_audio_file.is_file():
    display(Audio(str(preview_audio_file)))


In [ ]:
if "process" in globals() and process.poll() is None:
    raise RuntimeError("Worker vẫn đang chạy; theo dõi cell tiến độ, không mở lượt thứ hai.")
MODEL_CACHE = Path("/kaggle/working/voice-models")  # Có thể trỏ tới HF cache đã tải trong input
ALLOW_DOWNLOAD = True  # False khi dùng cache đã chuẩn bị và không có Internet
os.environ["CONTENT_BOT_VOICE_MODEL_DIR"] = str(MODEL_CACHE)
os.environ["VOICE_ALLOW_DOWNLOAD"] = "1" if ALLOW_DOWNLOAD else "0"
os.environ["PYTHONUTF8"] = "1"
log = (ROOT / "worker.log").open("w")
process = subprocess.Popen([PYTHON, str(ROOT / "worker.py"), "run", str(ROOT)], stdout=log, stderr=log)
print("Worker PID:", process.pid)


In [ ]:
import time
from IPython.display import clear_output
while process.poll() is None:
    progress_file = ROOT / "progress.json"
    if progress_file.exists():
        clear_output(wait=True)
        progress = json.loads(progress_file.read_text())
        print(progress["message"], "Xong:", len(progress["completed"]), "Lỗi:", len(progress["failed"]))
    time.sleep(5)
log.close()
print("Exit:", process.returncode)
if process.returncode:
    print((ROOT / "worker.log").read_text()[-4000:])


Cell dưới có thể chạy bất cứ lúc nào để lưu các đoạn đã hoàn thành. Muốn dừng sau đoạn hiện tại, ghi `{"action":"pause"}` vào `ROOT / "control.json"`, đợi worker kết thúc rồi đóng gói.


In [ ]:
import hashlib
OUTPUT = Path("/kaggle/working/voiceover-results.zip")
TEMP_OUTPUT = OUTPUT.with_suffix(".part.zip")
with zipfile.ZipFile(TEMP_OUTPUT, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ROOT / "manifest.json", "manifest.json")
    if (ROOT / "progress.json").exists():
        z.write(ROOT / "progress.json", "progress.json")
    if (ROOT / "environment.txt").exists():
        z.write(ROOT / "environment.txt", "environment.txt")
    for wav in (ROOT / "assets").glob("*.wav"):
        sidecar = wav.with_suffix(".json")
        if sidecar.exists():
            metadata = json.loads(sidecar.read_text())
            if metadata.get("checksum") != hashlib.sha256(wav.read_bytes()).hexdigest():
                print("Bỏ qua audio chưa hợp lệ:", wav.name)
                continue
            z.write(wav, "assets/" + wav.name)
            z.write(sidecar, "assets/" + sidecar.name)
TEMP_OUTPUT.replace(OUTPUT)
print("Đã lưu", OUTPUT, "— tải về hoặc Save Version với output trước khi hết phiên.")
from IPython.display import FileLink
display(FileLink(str(OUTPUT)))
